In [55]:
!pip install -U -q google-generativeai langchain langchain-google-genai langchain_community pypdf chromadb

In [56]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from IPython.display import Markdown as md

import warnings
warnings.filterwarnings('ignore')

In [57]:
from google.colab import userdata

In [58]:
import google.generativeai as genai

GEMINI = userdata.get('GEMINI')
api_key = GEMINI
genai.configure(api_key=api_key)


In [59]:
chat_model = ChatGoogleGenerativeAI(google_api_key=GEMINI,
                                   model="gemini-3.6-flash")

In [60]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Knowledge_Base_Wisata_Kuliner_Binjai.pdf")
pages = loader.load_and_split()

In [61]:
pages[0].page_content

'Panduan Destinasi Wisata & Kuliner Kota Binjai\nKnowledge Base Document | RAG Smart Travel & Culinary Recommendation Agent\nDeskripsi Dokumen: Dokumen ini dirancang khusus sebagai basis pengetahuan (Knowledge Base) untuk di-index\nke dalam Vector DB (ChromaDB/FAISS) pada proyek chatbot RAG. Berisi informasi terstruktur mengenai objek\nwisata alam, tempat rekreasi keluarga, kuliner legendaris, serta ulasan tempat makan khas Kota Binjai, Sumatera\nUtara. \n1. Profil Singkat Kota Binjai\nKota Binjai merupakan salah satu kota terdekat dari Medan, Sumatera Utara (berjarak ±22 km). Dikenal secara nasional\nsebagai "Kota Rambutan", Binjai tumbuh menjadi pusat rekreasi keluarga, wisata alam tepi sungai, serta surga kuliner\npercampuran budaya Melayu, Jawa, Tionghoa (Peranakan), dan Batak. \n2. Rekomendasi Destinasi Wisata Populer & Hits\nWISATA ALAM & KULINER\n1. Sawah Lukis Binjai\nLokasi: Jl. Pertamina, Kel. Cengkeh Turi, Kec. Binjai Utara | Jam Buka: 09.00 - 19.00 WIB\nTiket Masuk: Gratis 

In [62]:
len(pages)

3

In [63]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [64]:
from langchain_text_splitters import NLTKTextSplitter

# Misalkan kamu memiliki dokumen seperti berikut
simple_doc = """Panduan Destinasi Wisata & Kuliner Kota Binjai"""
print('panjang total karakter:',len(simple_doc),'\n')
# Membuat objek NLTKTextSplitter dengan ukuran chunk dan overlap
text_splitter = NLTKTextSplitter(separator='\n\n',chunk_size=67, chunk_overlap=10) #default separator='\n\n'

# Memecah dokumen menjadi beberapa chunk
chunks = text_splitter.split_text(simple_doc)
print(chunks,'\n')

# Menampilkan hasil chunk
for i, chunk in enumerate(chunks):
    #panjang karakter
    print(f"Panjang chunk {i+1}: {len(chunk)} karakter")
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 50)


panjang total karakter: 46 

['Panduan Destinasi Wisata & Kuliner Kota Binjai'] 

Panjang chunk 1: 46 karakter
Chunk 1:
Panduan Destinasi Wisata & Kuliner Kota Binjai
--------------------------------------------------


In [65]:
from langchain_text_splitters import NLTKTextSplitter

text_splitter = NLTKTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = text_splitter.split_documents(pages)

print(len(chunks))

print(type(chunks[0]))

14
<class 'langchain_core.documents.base.Document'>


In [66]:
# Mengecek hasil pemecahan menjadi chunk
print(f"Jumlah chunks yang dihasilkan: {len(chunks)}")
print(f"Tipe data chunk pertama: {type(chunks[0])}")

Jumlah chunks yang dihasilkan: 14
Tipe data chunk pertama: <class 'langchain_core.documents.base.Document'>


In [67]:
# Menampilkan chunk pertama
# panjang chunk
print(f"Panjang chunk pertama: {len(chunks[0].page_content)} karakter")
print("\nContoh chunk pertama:")
print(chunks[0].page_content)  # Memastikan bahwa setiap chunk memiliki konten

Panjang chunk pertama: 456 karakter

Contoh chunk pertama:
Panduan Destinasi Wisata & Kuliner Kota Binjai
Knowledge Base Document | RAG Smart Travel & Culinary Recommendation Agent
Deskripsi Dokumen: Dokumen ini dirancang khusus sebagai basis pengetahuan (Knowledge Base) untuk di-index
ke dalam Vector DB (ChromaDB/FAISS) pada proyek chatbot RAG.

Berisi informasi terstruktur mengenai objek
wisata alam, tempat rekreasi keluarga, kuliner legendaris, serta ulasan tempat makan khas Kota Binjai, Sumatera
Utara.

1.


In [68]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(google_api_key=api_key, model="gemini-embedding-2", output_dimensionality=768)

In [69]:
from langchain_community.vectorstores import Chroma

# Sematkan setiap chunk dan muat ke dalam chromadb
db = Chroma.from_documents(chunks, embedding_model, persist_directory="./chroma_db_")

# Menyimpan perubahan ke disk
db.persist()

In [70]:
# mengatur koneksi untuk menghubungkan ke ChromaDB
db_connection = Chroma(persist_directory="./chroma_db_", embedding_function=embedding_model)

In [71]:
# Mengonversi koneksi Chroma menjadi objek retriever untuk pencarian dokumen berbasis vektor
retriever = db_connection.as_retriever(search_kwargs={"k": 10})

print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [72]:
check_response = retriever.invoke("Ada berapa tempat wisata terfavorit di Binjai")
len(check_response)

10

In [73]:
md(check_response[0].page_content)

Knowledge Base RAG - Wisata & Kuliner Binjai Halaman 2

In [74]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

In [75]:
# Membuat template pesan untuk sistem dan pesan pengguna
chat_template = ChatPromptTemplate.from_messages([
    # System Message Prompt Template
    SystemMessage(content="""Anda adalah AI yang dapat menjawab pertanyaan berdasarkan konteks dan pertanyaan dari user.
                 Anda harus menjawab pertanyaan user, berdasarkan konteks"""),

    # Human Message Prompt Template
    HumanMessagePromptTemplate.from_template("""Jawab pertanyaan berikut berdasarkan konteks.
    konteks: {context}
    pertanyaan: {question}
    jawaban: """)
])

In [76]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [77]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [78]:
# Setelah dokumen diambil oleh retriever dan di-format
formatted_docs = format_docs(chunks)  # Format dokumen yang dihasilkan dari text chunks

# Cetak hasil format
print("Hasil Format Docs:")
print(formatted_docs)  # Menampilkan hasil setelah dokumen diformat

Hasil Format Docs:
Panduan Destinasi Wisata & Kuliner Kota Binjai
Knowledge Base Document | RAG Smart Travel & Culinary Recommendation Agent
Deskripsi Dokumen: Dokumen ini dirancang khusus sebagai basis pengetahuan (Knowledge Base) untuk di-index
ke dalam Vector DB (ChromaDB/FAISS) pada proyek chatbot RAG.

Berisi informasi terstruktur mengenai objek
wisata alam, tempat rekreasi keluarga, kuliner legendaris, serta ulasan tempat makan khas Kota Binjai, Sumatera
Utara.

1.

1.

Profil Singkat Kota Binjai
Kota Binjai merupakan salah satu kota terdekat dari Medan, Sumatera Utara (berjarak ±22 km).

Dikenal secara nasional
sebagai "Kota Rambutan", Binjai tumbuh menjadi pusat rekreasi keluarga, wisata alam tepi sungai, serta surga kuliner
percampuran budaya Melayu, Jawa, Tionghoa (Peranakan), dan Batak.

2.

Rekomendasi Destinasi Wisata Populer & Hits
WISATA ALAM & KULINER
1.

Sawah Lukis Binjai
Lokasi: Jl.

Pertamina, Kel.

Cengkeh Turi, Kec.

Sawah Lukis Binjai
Lokasi: Jl.

Pertamina, Kel.

In [79]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | chat_template
    | chat_model
    | output_parser
)


In [80]:
response = rag_chain.invoke("""Beritahu saya tempat makan enak di Binjai""")

response

'Berdasarkan konteks yang diberikan, berikut adalah tempat makan/kuliner yang terdapat di Binjai:\n\n1. **Tau Kua Heci (Apo Brahrang)**\n   * **Lokasi:** Jl. Anggur No. 64x, Brahrang, Binjai\n   * **Kisaran Harga:** Rp25.000 - Rp40.000/porsi\n   * **Deskripsi:** Hidangan khas Peranakan Tionghoa-Binjai yang berisi irisan tahu goreng, bakwan udang renyah, kangkung rebus, tauge, dan daging/udang yang disiram kuah saus merah kental dengan rasa asam, manis, dan sedikit gurih.\n\n2. **Sawah Lukis Binjai** (Wisata Alam & Kuliner)\n   * **Lokasi:** Jl. Pertamina, Kel. Cengkeh Turi, Kec. Binjai Utara\n   * **Jam Buka:** 09.00 - 19.00 WIB\n   * **Deskripsi:** Tempat wisata alam dan kuliner berupa hamparan sawah dengan saung/pondok bambu. Tiket masuk gratis, tetapi pengunjung wajib memesan makanan/minuman di tempat ini.'

In [81]:
md(response)

Berdasarkan konteks yang diberikan, berikut adalah tempat makan/kuliner yang terdapat di Binjai:

1. **Tau Kua Heci (Apo Brahrang)**
   * **Lokasi:** Jl. Anggur No. 64x, Brahrang, Binjai
   * **Kisaran Harga:** Rp25.000 - Rp40.000/porsi
   * **Deskripsi:** Hidangan khas Peranakan Tionghoa-Binjai yang berisi irisan tahu goreng, bakwan udang renyah, kangkung rebus, tauge, dan daging/udang yang disiram kuah saus merah kental dengan rasa asam, manis, dan sedikit gurih.

2. **Sawah Lukis Binjai** (Wisata Alam & Kuliner)
   * **Lokasi:** Jl. Pertamina, Kel. Cengkeh Turi, Kec. Binjai Utara
   * **Jam Buka:** 09.00 - 19.00 WIB
   * **Deskripsi:** Tempat wisata alam dan kuliner berupa hamparan sawah dengan saung/pondok bambu. Tiket masuk gratis, tetapi pengunjung wajib memesan makanan/minuman di tempat ini.

In [82]:
# Instalasi dependensi yang dibutuhkan
!pip install -q langchain PyPDF2 #python-dotenv

In [83]:
import os
import io
import PyPDF2
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
# from dotenv import load_dotenv
from google.colab import files
from IPython.display import Markdown as md

In [84]:
# Membuka dan membaca file PDF
with open('/content/Knowledge_Base_Wisata_Kuliner_Binjai.pdf', "rb") as file:
    pdf_reader = PyPDF2.PdfReader(file)
    pdf_pages = pdf_reader.pages

    # Mengekstrak teks dari semua halaman
    context = "\n\n".join(page.extract_text() for page in pdf_pages)

In [85]:
context

'Panduan Destinasi Wisata & Kuliner Kota Binjai\nKnowledge Base Document | RAG Smart Travel & Culinary Recommendation Agent\nDeskripsi Dokumen:  Dokumen ini dirancang khusus sebagai basis pengetahuan (Knowledge Base) untuk di-index\nke dalam Vector DB (ChromaDB/FAISS) pada proyek chatbot RAG. Berisi informasi terstruktur mengenai objek\nwisata alam, tempat rekreasi keluarga, kuliner legendaris, serta ulasan tempat makan khas Kota Binjai, Sumatera\nUtara. \n1. Profil Singkat Kota Binjai\nKota Binjai merupakan salah satu kota terdekat dari Medan, Sumatera Utara (berjarak ±22 km). Dikenal secara nasional\nsebagai "Kota Rambutan" , Binjai tumbuh menjadi pusat rekreasi keluarga, wisata alam tepi sungai, serta surga kuliner\npercampuran budaya Melayu, Jawa, Tionghoa (Peranakan), dan Batak. \n2. Rekomendasi Destinasi Wisata Populer & Hits\nWISATA ALAM & KULINER\n1. Sawah Lukis Binjai\nLokasi:  Jl. Pertamina, Kel. Cengkeh Turi, Kec. Binjai Utara | Jam Buka:  09.00 - 19.00 WIB\nTiket Masuk:  Gr

In [86]:
# Memecah teks menjadi potongan-potongan kecil
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_text(context)

In [87]:
print(len(texts))

print(type(texts[0]))

14
<class 'str'>


In [88]:
texts[0]

'Panduan Destinasi Wisata & Kuliner Kota Binjai\nKnowledge Base Document | RAG Smart Travel & Culinary Recommendation Agent\nDeskripsi Dokumen:  Dokumen ini dirancang khusus sebagai basis pengetahuan (Knowledge Base) untuk di-index\nke dalam Vector DB (ChromaDB/FAISS) pada proyek chatbot RAG. Berisi informasi terstruktur mengenai objek\nwisata alam, tempat rekreasi keluarga, kuliner legendaris, serta ulasan tempat makan khas Kota Binjai, Sumatera\nUtara. \n1. Profil Singkat Kota Binjai'

In [89]:
# Membuat embeddings untuk potongan-potongan teks
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2",google_api_key=api_key)

vector_index = Chroma.from_texts(texts, embeddings).as_retriever(search_kwargs={"k": 20})

In [90]:
vector_index

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7b5871b4c920>, search_kwargs={'k': 20})

In [91]:
print(type(vector_index))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [92]:
# Mendapatkan pertanyaan dari pengguna
user_question = input("Tanyakan pertanyaan: ")

# Mendapatkan dokumen relevan untuk pertanyaan pengguna
docs = vector_index.invoke(user_question)

Tanyakan pertanyaan: Apa yang ada di binjai


In [93]:
# Mendefinisikan template prompt
prompt_template = """
Jawablah pertanyaan ini dengan se-detail mungkin dari konteks yang diberikan,
pastikan untuk memberikan semua detail, jika jawaban tidak ada dalam
konteks yang diberikan cukup katakan, "jawaban tidak tersedia dalam konteks",
jangan memberikan jawaban yang salah\n\n
Konteks:\n {context}?\n
Pertanyaan: \n{question}\n
Jawaban:
"""

# Membuat prompt
prompt = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])

# Memuat QA chain
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=api_key)
chain = load_qa_chain(model, chain_type="stuff", prompt=prompt)

# Mendapatkan jawaban dari model
response = chain({"input_documents": docs, "question": user_question}, return_only_outputs=True)

# Menampilkan jawaban
print("\nJawaban:")
md(response['output_text'])


Jawaban:


Berdasarkan konteks yang diberikan, berikut adalah detail mengenai apa saja yang ada di Kota Binjai:

**1. Profil Singkat & Julukan**
* Kota Binjai merupakan salah satu kota terdekat dari Medan, Sumatera Utara (berjarak ±22 km).
* Dikenal secara nasional sebagai **"Kota Rambutan"**.
* Merupakan pusat rekreasi keluarga, wisata alam tepi sungai, serta surga kuliner percampuran budaya Melayu, Jawa, Tionghoa (Peranakan), dan Batak.

**2. Destinasi Wisata Populer & Hits**
* **Wisata Alam & Kuliner:**
  * **Sawah Lukis Binjai:** Berlokasi di Jl. Pertamina, Kel. Cengkeh Turi, Kec. Binjai Utara (Jam Buka: 09.00 - 19.00 WIB, Tiket Masuk: Gratis dengan kewajiban memesan makanan/minuman). Menawarkan hamparan sawah hijau asri, saung/pondok bambu estetik, sangat instagramable, dan memiliki fasilitas penyewaan ban pelampung, saung tepi sungai, warung makan lokal, kamar ganti, area makan outdoor, spot foto lukisan sawah, area parkir luas, musala, serta toilet.
* **Rekreasi & Foto:**
  * **Wisata Jona Garden (WJG):** Berlokasi di Emplasmen Kwala Mencirim, Kec. Sei Bingai (Perbatasan Binjai) (Jam Buka: 08.00 - 22.00 WIB, Tiket Masuk: Rp10.000 - Rp15.000 untuk Weekday/Weekend). Taman rekreasi keluarga skala besar berkonsep modern yang menyajikan pemandangan taman bunga, berbagai wahana permainan air, hingga spot foto replika landmark dunia. Fasilitasnya meliputi waterpark, kolam renang anak & dewasa, flying fox, wahana ATV, resto & cafe, serta wisma/penginapan.
* **Wisata Alam & River Tubing:**
  * **Namo Sira-Sira & Pantai Florida Binjai:** Berlokasi di Kec. Sei Bingai, Kab. Langkat (Akses Utama via Binjai) (Jam Buka: 08.00 - 18.00 WIB, Tiket Masuk: Rp10.000 - Rp20.000). Pemandian alam dengan aliran air sungai yang sangat jernih dan segar bersumber dari pegunungan, dikelilingi pepohonan rindang dan bebatuan alami, sangat cocok untuk berenang dan bermain air.
* **Rekreasi Kota:**
  * **Tanah Lapang Merdeka Binjai & Taman Kota:** Berlokasi di Jl. Jenderal Sudirman, Binjai Kota (Jam Buka: 24 Jam, Tiket Masuk: Gratis). Alun-alun pusat Kota Binjai yang menjadi pusat kegiatan masyarakat, olahraga, serta rekreasi malam hari, yang di sekitarnya terdapat deretan wahana permainan outdoor anak-anak dan pusat kuliner malam.

**3. Destinasi Kuliner Legendaris & Tempat Makan Favorit**
* **Kuliner Ikonik / Oleh-Oleh:**
  * **Tahu Balek / Tahu Balik Pondok Surya:** Berlokasi di Jl. Ade Irma Suryani Gg. Anggrek No. 6, Binjai (Kisaran Harga: Rp15.000 - Rp35.000/porsi). Kuliner paling ikonik dari Binjai berupa tahu putih yang dibalik sehingga bagian dalam menjadi renyah di luar, diisi dengan adonan daging bakso ayam/sapi yang padat dan gurih, serta disajikan dengan sambal kecap pedas gurih.
  * *Catatan:* Terdapat juga hidangan khas Peranakan Tionghoa-Binjai (tanpa disebutkan nama spesifik tempatnya, namun berada di Jl. Anggur No. 64x, Brahrang, Binjai dengan kisaran harga Rp25.000 - Rp40.000/porsi) yang terdiri dari irisan tahu goreng, bakwan udang renyah, kangkung rebus, tauge, dan daging/udang yang disiram kuah saus merah kental dengan cita rasa asam, manis, dan sedikit gurih.
* **Street Food Malam:**
  * **Pasar Kaget Binjai:** Berlokasi di Jl. Ahmad Yani, Binjai Kota (Jam Buka: 18.00 - 24.00 WIB, Harga: Sangat terjangkau Rp10.000 - Rp30.000). Surga kuliner malam kaki lima yang menyediakan ratusan jenis hidangan mulai dari Martabak, Sate Padang, Nasi Goreng, Mie Sop, Dimsum, hingga seafood dan camilan tradisional Jawa-Melayu.